# Multi-Agentes

Conforme um agente acumula muitas ferramentas, fica dificil para o modelo decidir qual usar em cada situacao. A qualidade das respostas cai porque o modelo precisa escolher entre dezenas de opcoes.

A solucao e dividir as responsabilidades entre **agentes especializados**, cada um com poucas tools e um escopo claro, coordenados por um **agente principal**. O padrao e simples: cada sub-agente e encapsulado dentro de uma tool que o coordenador pode chamar.

Neste notebook vamos criar dois sub-agentes (um financeiro e um pesquisador) e um coordenador que delega tarefas conforme a pergunta do usuario.

In [ ]:
from dotenv import load_dotenv

load_dotenv()

## Agentes especializados

Vamos criar dois sub-agentes, cada um com uma unica tool e um system prompt focado.

O **agente financeiro** faz conversao de moeda entre BRL, USD e EUR.

In [ ]:
from langchain.tools import tool

@tool
def converter_moeda(valor: float, de: str, para: str) -> str:
    """Converte valores entre BRL, USD e EUR usando taxas fixas."""
    taxas_para_brl = {"BRL": 1.0, "USD": 5.80, "EUR": 6.30}
    valor_brl = valor * taxas_para_brl[de]
    resultado = valor_brl / taxas_para_brl[para]
    return f"{valor:.2f} {de} = {resultado:.2f} {para}"

In [ ]:
from langchain.agents import create_agent

agente_financeiro = create_agent(
    model="gpt-4.1-nano",
    tools=[converter_moeda],
    system_prompt="Voce e um especialista financeiro. Use sua ferramenta para fazer conversoes de moeda."
)

O **agente pesquisador** busca informacoes atualizadas na web usando Tavily.

In [ ]:
from typing import Dict, Any
from tavily import TavilyClient

tavily_client = TavilyClient()

@tool
def buscar_na_web(query: str) -> Dict[str, Any]:
    """Busca informacoes atualizadas na internet."""
    return tavily_client.search(query)

In [ ]:
agente_pesquisador = create_agent(
    model="gpt-4.1-nano",
    tools=[buscar_na_web],
    system_prompt="Voce e um pesquisador de mercado. Use a busca na web para encontrar informacoes atualizadas."
)

## Encapsulando sub-agentes em tools

Para que o coordenador possa delegar tarefas, cada sub-agente precisa ser encapsulado em uma tool. A tool recebe a pergunta do usuario, invoca o sub-agente internamente, e retorna a resposta.

In [ ]:
from langchain.messages import HumanMessage

@tool
def consultar_financeiro(pergunta: str) -> str:
    """Consulta o especialista financeiro para conversoes de moeda entre BRL, USD e EUR."""
    resposta = agente_financeiro.invoke({"messages": [HumanMessage(content=pergunta)]})
    return resposta["messages"][-1].content

@tool
def pesquisar_mercado(pergunta: str) -> str:
    """Pesquisa informacoes atualizadas sobre o mercado usando a web."""
    resposta = agente_pesquisador.invoke({"messages": [HumanMessage(content=pergunta)]})
    return resposta["messages"][-1].content

Cada tool esconde a complexidade do sub-agente. O coordenador so precisa saber o que a tool faz (pela descricao), nao como ela faz internamente.

## Agente coordenador

O coordenador recebe as wrapper tools e decide para qual especialista delegar cada pergunta.

In [ ]:
coordenador = create_agent(
    model="gpt-4.1-nano",
    tools=[consultar_financeiro, pesquisar_mercado],
    system_prompt=(
        "Voce e um assistente que coordena especialistas. "
        "Para questoes de conversao de moeda, delegue ao especialista financeiro. "
        "Para questoes que exigem informacoes atualizadas, delegue ao pesquisador de mercado."
    )
)

## Testando a delegacao

Vamos fazer duas perguntas distintas para verificar que o coordenador delega para o sub-agente correto.

In [ ]:
resposta = coordenador.invoke(
    {"messages": [HumanMessage(content="Quanto e 10 mil reais em dolares?")]}
)

print(resposta["messages"][-1].content)

In [ ]:
resposta = coordenador.invoke(
    {"messages": [HumanMessage(content="Como esta o mercado imobiliario em Sao Paulo atualmente?")]}
)

print(resposta["messages"][-1].content)

## Inspecionando a delegacao

Podemos ver nas mensagens qual sub-agente foi chamado pelo coordenador. A `tool_calls` na AIMessage mostra a decisao do modelo.

In [ ]:
from pprint import pprint

pprint(resposta["messages"])

O fluxo e o mesmo que ja vimos com tools simples: o modelo gera uma tool call, o LangChain executa a tool (que internamente invoca o sub-agente), e o resultado volta para o modelo formular a resposta final.

Esse padrao de **sub-agente encapsulado em tool** e a forma mais simples e eficaz de construir sistemas multi-agente. Cada especialista tem seu escopo bem definido, e o coordenador decide a quem delegar. No proximo notebook, vamos combinar isso com estado para criar um sistema completo.